# Checkpointing: Truncating Lineage and Durable State Preservation

**Module**: 11_fault_tolerance_and_reliability  
**Notebook**: 04_checkpointing.ipynb  
**Environment**: Apache Spark 4.2.1 · Standalone Cluster · PySpark · Python 3.13 · Java 21  

---

## Learning Objectives

- **What will I learn?**: How checkpointing saves materialized data to durable storage and completely severs the upstream lineage graph.
- **Why does this topic exist?**: Recursive or iterative algorithms generate massive DAGs that cause JVM StackOverflowError unless checkpointed.
- **What problem does it solve?**: JVM crashing due to 500-step iterative DAG lineage.
- **Where is it used in real-world Data Engineering?**: Iterative graph algorithms, ML training, and streaming recovery.

---

## 2. Checkpoint vs Persist
- `persist()`: Keeps data in memory/disk; **retains full lineage**.
- `checkpoint()`: Saves to reliable storage; **truncates lineage completely**.

### 1. Initialize SparkSession on Standalone Cluster

We connect to `spark://spark-master:7077` with the driver configured to bind to container hostname `jupyter`.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
import time

spark = (
    SparkSession.builder
    .appName("Checkpointing_Lab")
    .master("spark://spark-master:7077")
    .config("spark.driver.host", "jupyter")
    .config("spark.driver.bindAddress", "0.0.0.0")
    .config("spark.eventLog.enabled", "true")
    .config("spark.eventLog.dir", "file:///opt/spark-events")
    .config("spark.sql.warehouse.dir", "file:///opt/spark-data/warehouse")
    .config("spark.hadoop.fs.permissions.umask-mode", "000")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

print(f" SparkSession initialized successfully!")
print(f"Spark Version:      {spark.version}")
print(f"Master URL:         {spark.sparkContext.master}")
print(f"App Name:           {spark.sparkContext.appName}")
print(f"Default Parallelism: {spark.sparkContext.defaultParallelism}")
print(f"Spark UI available at: http://localhost:4040")

In [ ]:
chk_dir = "/opt/spark-data/warehouse/rdd_checkpoint_dir"
os.system(f"rm -rf {chk_dir}")
spark.sparkContext.setCheckpointDir(f"file://{chk_dir}")

# Build RDD and checkpoint
rdd = spark.sparkContext.parallelize(range(20), 2).map(lambda x: x * 3)
rdd.checkpoint()

print("Lineage before Action:")
print(rdd.toDebugString().decode('utf-8'))

# Action triggers materialization and lineage truncation
rdd.count()

print("\nLineage AFTER Action (Notice: Lineage is TRUNCATED!):")
print(rdd.toDebugString().decode('utf-8'))

---

## Experiment: Verifying Lineage Truncation

- **1. What are we testing?**: Verify that toDebugString() points directly to CheckpointRDD after action.
- **2. What do we expect?**: Original parallelize and map steps are completely severed.

In [ ]:
print('Is RDD checkpointed?:', rdd.isCheckpointed())

**3. What happened?**: isCheckpointed returned True; parent steps vanished from debug string.  
**4. Why did it happen?**: Spark wrote the materialized partition files to disk and reset the parent pointer.  
**5. Practical Engineering Lesson**: Checkpointing cuts the lineage graph, freeing JVM memory from deep recursion.

---

## Practical Exercises

### Exercise 1 (Beginner): Set Checkpoint Dir
**Problem Statement**: Set checkpoint directory on SparkContext.  
**Input / Context**: setCheckpointDir

In [ ]:
# Write your solution for Exercise 1 here

### Exercise 2 (Intermediate): DataFrame Checkpoint
**Problem Statement**: Checkpoint a DataFrame using df.checkpoint().  
**Input / Context**: Provided dataset

In [ ]:
# Write your solution for Exercise 2 here

### Exercise 3 (Advanced): Eager vs Non-Eager Checkpoint
**Problem Statement**: Explain df.checkpoint(eager=False).  
**Input / Context**: Provided dataset

In [ ]:
# Write your solution for Exercise 3 here

---

## Reference Solutions

<details>
<summary><b>Click to expand full solutions</b></summary>

#### Solution 1 (Set Checkpoint Dir):
```python
spark.sparkContext.setCheckpointDir('file:///opt/spark-data/warehouse/chk')
```

#### Solution 2 (DataFrame Checkpoint):
```python
df = spark.range(10); df_c = df.checkpoint(); print('DF checkpointed:', df_c.isLocal)
```

#### Solution 3 (Eager vs Non-Eager Checkpoint):
```python
print('eager=True executes immediately; eager=False waits for downstream action.')
```

</details>

---

## Module Summary & Interview Guide

### Key Takeaways
- **Lineage Cut**: Checkpointing permanently severs parent dependencies.
- **Durable Storage**: Saved to reliable disk/HDFS.

### Essential Technical Terms
- `CheckpointRDD`: RDD reading directly from checkpoint files.
- `setCheckpointDir`: Registers checkpoint storage path.

### Common Engineering Mistakes
- **Using persist instead of checkpoint for deep loops**: Causes JVM StackOverflowError.

### Interview Questions on This Topic
- **Q: What is the difference between persist() and checkpoint()?**
  - *A*: persist() keeps data in executor cache while retaining the lineage graph; checkpoint() writes data to durable storage and permanently severs the lineage graph.

### What to Study Next
Open 05_fault_tolerance_experiments.ipynb

---

### Tear Down SparkSession
Always close your session when finishing to release executor threads and driver ports.

In [ ]:
spark.stop()
print('SparkSession successfully stopped.')